
# 05\_modele\_demografike\_epidemike\_trafiku

## Modele demografike, epidemike dhe të trafikut

Ky notebook është pjesë e kursit **Modelim në Fizikë** dhe ka për qëllim të tregojë se si mjetet e modelimit matematikor dhe numerik, të përdorura gjerësisht në fizikë, mund të zbatohen edhe në sisteme biologjike dhe sociale.  
Edhe pse këto sisteme nuk janë mekanike në kuptimin klasik, ato ndajnë me sistemet fizike shumë tipare thelbësore:

- evolucion në kohë,
- ndërveprime midis njësive përbërëse,
- jolinearitet,
- ekuilibra dhe stabilitet,
- shfaqje të sjelljeve kolektive emergjente.

Në këtë pjesë do të studiojmë tri klasa të rëndësishme modelesh:

1. **modelet demografike**, që përshkruajnë evolucionin e popullatave;
2. **modelet epidemike**, që modelojnë përhapjen e sëmundjeve infektive;
3. **modelet e trafikut**, që përshkruajnë lëvizjen kolektive të automjeteve.

Në secilin rast, do të kombinohen:

- formulimi matematikor,
- zgjidhja numerike,
- vizualizimi grafik,
- interpretimi fizik i rezultateve.

> **Qëllimi pedagogjik:** studenti të kuptojë se i njëjti aparat konceptual — ekuacione diferenciale, modele diskrete, simulime numerike — mund të shërbejë si gjuhë unifikuese për fenomene shumë të ndryshme.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 11



## 1. Hyrje në modelimin e sistemeve sociale dhe biologjike

Në fizikë jemi mësuar të modelojmë sisteme si grimcat, oshilatorët, fluidet ose fushat. Megjithatë, shumë prej ideve të fizikës matematikore mund të transferohen edhe te sistemet ku njësitë përbërëse janë:

- organizma biologjikë,
- individë në një popullatë,
- automjete në një rrjet rrugor,
- subjekte që ndërveprojnë sipas rregullave të caktuara.

Këto sisteme përfaqësojnë shpesh **sisteme komplekse**, në të cilat sjellja makroskopike nuk mund të kuptohet plotësisht vetëm nga sjellja e një njësie të vetme.

### Disa pyetje themelore

- Si rritet një popullatë kur burimet janë të bollshme?
- Si stabilizohet ajo kur burimet bëhen të kufizuara?
- Kur shpërthen një epidemi dhe çfarë përcakton kulmin e saj?
- Si formohen kolonat dhe bllokimet në trafik, edhe pa një pengesë të jashtme?

Nga pikëpamja matematikore, këto pyetje adresohen duke ndërtuar modele në të cilat përcaktojmë:

- variablat e gjendjes,
- parametrat kontrollues,
- ligjet e evolucionit në kohë.

Në këtë notebook do të përdorim si **modele të vazhdueshme** (ekuacione diferenciale ordinare) ashtu edhe **modele diskrete** (automate qelizore dhe rregulla të thjeshta për agjentë individualë).



## 2. Modelet demografike

Modelet demografike janë ndër modelet më të hershme dhe më klasike në biologjinë matematikore.  
Objekti kryesor është funksioni

\[
N(t),
\]

i cili përfaqëson madhësinë e popullatës në kohë.

Në varësi të hipotezave mbi lindjet, vdekjet dhe kufizimet e mjedisit, mund të ndërtojmë modele me kompleksitet të ndryshëm.  
Ne do të fillojmë me dy raste themelore:

- modeli eksponencial,
- modeli logjistik.



### 2.1 Modeli eksponencial

Modeli eksponencial supozon se norma e ndryshimit të popullatës është proporcionale me vetë popullatën:

\[
\frac{dN}{dt} = rN,
\]

ku:

- \(N(t)\) është popullata,
- \(r\) është norma neto e rritjes.

Nëse \(r>0\), popullata rritet; nëse \(r<0\), ajo bie.  
Zgjidhja analitike është:

\[
N(t) = N_0 e^{rt}.
\]

Ky model është i rëndësishëm si pikënisje, por ka një kufizim të madh: ai nuk merr parasysh burimet e kufizuara.


In [ ]:

# Modeli eksponencial: zgjidhja analitike
r = 0.18
N0 = 20
t = np.linspace(0, 40, 500)

N = N0 * np.exp(r * t)

plt.figure()
plt.plot(t, N, linewidth=2)
plt.xlabel("Koha")
plt.ylabel("Popullata N(t)")
plt.title("Rritja eksponenciale e popullatës")
plt.grid(True, alpha=0.3)
plt.show()



#### Interpretim

Modeli eksponencial është i përshtatshëm për fazat e hershme të rritjes, kur:

- burimet janë praktikisht të pakufizuara,
- konkurrenca është e papërfillshme,
- popullata është ende larg kufirit të mjedisit.

Në aspekt fizik dhe sistemor, ky model përfaqëson një **feedback pozitiv të pastër**: sa më e madhe popullata, aq më i madh ritmi absolut i rritjes.



### 2.2 Modeli logjistik

Për të marrë parasysh kufizimet e mjedisit, futet **kapaciteti mbajtës** \(K\), i cili përfaqëson madhësinë maksimale të popullatës që mund të mbështetet në mënyrë të qëndrueshme.

Ekuacioni logjistik është:

\[
\frac{dN}{dt} = rN\left(1 - \frac{N}{K}\right).
\]

Ky term jolinear \(\left(1-\frac{N}{K}\right)\) modifikon rritjen:

- kur \(N \ll K\), sjellja është pothuajse eksponenciale;
- kur \(N \to K\), rritja ngadalësohet;
- kur \(N = K\), sistemi arrin ekuilibrin.

Ky model është shumë i rëndësishëm sepse paraqet një prej rasteve më themelore të **feedback-ut negativ stabilizues**.


In [ ]:

def logistic_rhs(t, y, r, K):
    N = y[0]
    return [r * N * (1 - N / K)]

r = 0.25
K = 150
N0 = [8]

t_eval = np.linspace(0, 60, 600)
sol = solve_ivp(logistic_rhs, (t_eval[0], t_eval[-1]), N0, t_eval=t_eval, args=(r, K))

plt.figure()
plt.plot(sol.t, sol.y[0], label="Modeli logjistik", linewidth=2)
plt.axhline(K, linestyle="--", label="Kapaciteti mbajtës K")
plt.xlabel("Koha")
plt.ylabel("Popullata N(t)")
plt.title("Rritja logjistike")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



#### Krahasim: modeli eksponencial kundrejt modelit logjistik

Ky krahasim është didaktikisht shumë i rëndësishëm: ai tregon se një modifikim relativisht i vogël në ekuacionin diferencial — shtimi i termit jolinear — ndryshon tërësisht sjelljen afatgjatë të sistemit.


In [ ]:

# Krahasim midis modelit eksponencial dhe logjistik
r = 0.2
K = 120
N0 = 10
t = np.linspace(0, 50, 500)

N_exp = N0 * np.exp(r * t)

sol_log = solve_ivp(
    logistic_rhs,
    (t[0], t[-1]),
    [N0],
    t_eval=t,
    args=(r, K)
)

plt.figure()
plt.plot(t, N_exp, label="Eksponencial", linewidth=2)
plt.plot(sol_log.t, sol_log.y[0], label="Logjistik", linewidth=2)
plt.axhline(K, linestyle="--", label="K")
plt.xlabel("Koha")
plt.ylabel("Popullata")
plt.title("Krahasimi i dy modeleve demografike")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



#### Koment fizik

- **Modeli eksponencial** nuk ka një shkallë natyrore saturimi.
- **Modeli logjistik** përfshin konkurrencën e brendshme dhe prodhon një gjendje të qëndrueshme.
- Në terma të sistemeve dinamike, modeli logjistik ka një pikë fikse tërheqëse te \(N=K\), për \(r>0\).

Këto ide lidhen ngushtë me nocionet e **stabilitetit**, **pikave fikse** dhe **jolinearitetit**, të cilat i kemi hasur edhe në sistemet fizike.



## 3. Modelet epidemike

Modelet epidemike ndajnë popullatën në nënklasa ose **kompartimente**.  
Në vend që të ndjekim çdo individ veçmas, ne përshkruajmë fraksionet ose numrat e individëve në secilën klasë.

Qasja është makroskopike dhe mesatare, në frymën e modeleve të fizikës statistikore.  
Dy modelet klasike që do të shohim janë:

- modeli **SIR**,
- modeli **SEIR**.



### 3.1 Modeli SIR

Në modelin SIR popullata ndahet në tri klasa:

- \(S(t)\): individët e ndjeshëm ndaj infeksionit,
- \(I(t)\): individët e infektuar,
- \(R(t)\): individët e larguar nga qarkullimi epidemik (të shëruar ose të imunizuar).

Ekuacionet janë:

\[
\frac{dS}{dt} = -\beta SI,
\]

\[
\frac{dI}{dt} = \beta SI - \gamma I,
\]

\[
\frac{dR}{dt} = \gamma I,
\]

ku:

- \(\beta\) është norma e transmetimit,
- \(\gamma\) është norma e rikuperimit.

Në këtë formulim, përplasjet efektive midis \(S\) dhe \(I\) japin infeksione të reja, ndërsa klasa \(I\) zbrazet me ritëm \(\gamma I\).


In [ ]:

def sir_rhs(t, y, beta, gamma):
    S, I, R = y
    dS = -beta * S * I
    dI = beta * S * I - gamma * I
    dR = gamma * I
    return [dS, dI, dR]

beta = 0.8
gamma = 0.2
y0 = [0.99, 0.01, 0.0]

t = np.linspace(0, 80, 1000)
sol_sir = solve_ivp(sir_rhs, (t[0], t[-1]), y0, t_eval=t, args=(beta, gamma))

S, I, R = sol_sir.y

plt.figure()
plt.plot(t, S, label="S", linewidth=2)
plt.plot(t, I, label="I", linewidth=2)
plt.plot(t, R, label="R", linewidth=2)
plt.xlabel("Koha")
plt.ylabel("Fraksioni i popullatës")
plt.title("Dinamika e modelit SIR")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



#### Numri bazë i riprodhimit

Një madhësi qendrore është

\[
R_0 = \frac{\beta}{\gamma},
\]

në formulimin e thjeshtuar me popullatë të normuar.  
Në terma fizikë dhe intuitivë, \(R_0\) mat aftësinë e infeksionit për t’u përhapur.

- Nëse \(R_0 > 1\), epidemia mund të rritet fillimisht.
- Nëse \(R_0 < 1\), infeksioni shuhet.

Ky është një shembull shumë i bukur se si një **parametër pa përmasa** kontrollon regjimin dinamik të sistemit.


In [ ]:

# Studim i ndikimit të beta-s në kulmin epidemik
gamma = 0.2
betas = [0.15, 0.30, 0.50, 0.80]
t = np.linspace(0, 80, 1000)

plt.figure()

for beta in betas:
    sol = solve_ivp(sir_rhs, (t[0], t[-1]), [0.99, 0.01, 0.0], t_eval=t, args=(beta, gamma))
    I = sol.y[1]
    plt.plot(t, I, label=fr"$\beta={beta:.2f}$")

plt.xlabel("Koha")
plt.ylabel("I(t)")
plt.title("Ndikimi i normës së transmetimit në modelin SIR")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



#### Interpretim

Nga grafiku vihet re se rritja e \(\beta\):

- e bën epideminë të shpërthejë më shpejt,
- e rrit kulmin e klasës së infektuar,
- e zhvendos kulmin në kohë më të hershme.

Ky është një shembull i qartë i ndjeshmërisë së sistemit ndaj parametrave kontrollues.



### 3.2 Modeli SEIR

Në shumë sëmundje ekziston një periudhë ndërmjet ekspozimit dhe infektivitetit.  
Për këtë arsye futet klasa \(E(t)\), e cila përfaqëson individët e ekspozuar, por ende jo infektivë.

Modeli SEIR shkruhet:

\[
\frac{dS}{dt} = -\beta SI,
\]

\[
\frac{dE}{dt} = \beta SI - \sigma E,
\]

\[
\frac{dI}{dt} = \sigma E - \gamma I,
\]

\[
\frac{dR}{dt} = \gamma I,
\]

ku \(\sigma\) kontrollon kalimin nga gjendja e ekspozuar në gjendjen infektive.


In [ ]:

def seir_rhs(t, y, beta, sigma, gamma):
    S, E, I, R = y
    dS = -beta * S * I
    dE = beta * S * I - sigma * E
    dI = sigma * E - gamma * I
    dR = gamma * I
    return [dS, dE, dI, dR]

beta = 0.9
sigma = 0.4
gamma = 0.2
y0 = [0.99, 0.0, 0.01, 0.0]

t = np.linspace(0, 100, 1200)
sol_seir = solve_ivp(seir_rhs, (t[0], t[-1]), y0, t_eval=t, args=(beta, sigma, gamma))

S, E, I, R = sol_seir.y

plt.figure()
plt.plot(t, S, label="S", linewidth=2)
plt.plot(t, E, label="E", linewidth=2)
plt.plot(t, I, label="I", linewidth=2)
plt.plot(t, R, label="R", linewidth=2)
plt.xlabel("Koha")
plt.ylabel("Fraksioni i popullatës")
plt.title("Dinamika e modelit SEIR")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



#### Krahasim konceptual midis SIR dhe SEIR

Futja e klasës \(E\) shton një **vonesë dinamike** në sistem.  
Kjo ka pasoja të rëndësishme:

- kulmi i infeksionit mund të vonohet,
- dinamika bëhet më realiste për sëmundjet me inkubacion,
- sistemi fiton një strukturë kohore më të pasur.

Nga këndvështrimi i modelimit, ky është një shembull tipik i rëndësisë së **zgjedhjes së variablave të gjendjes**.



## 4. Modele të trafikut

Trafiku rrugor është një sistem shumëagjentësh ku sjellja globale lind nga rregulla lokale të thjeshta.  
Ashtu si në shumë sisteme fizike, mund të shfaqen:

- tranzicione nga rrjedhja e lirë në bllokim,
- valë ngadalësimi që përhapen në drejtim të kundërt të lëvizjes,
- struktura kolektive që nuk janë të dukshme në nivelin e një automjeti të vetëm.

Ne do të konsiderojmë dy qasje:

- një model të tipit **car-following**,
- modelin diskret **Nagel–Schreckenberg**.



### 4.1 Modeli car-following

Në modelin car-following çdo automjet reagon ndaj automjetit përpara tij.  
Një formulim i thjeshtë është:

\[
\frac{dv_i}{dt} = a\,(v_{\mathrm{des}} - v_i) + b\,(x_{i+1} - x_i - d),
\]

ku:

- \(x_i\) është pozicioni i automjetit \(i\),
- \(v_i\) është shpejtësia,
- \(v_{\mathrm{des}}\) është shpejtësia e dëshiruar,
- \(d\) është distanca karakteristike e sigurisë,
- \(a\) dhe \(b\) janë parametra reagimi.

Ky model ka një interpretim të qartë fizik: shoferi përpiqet të arrijë një shpejtësi të dëshiruar, por korrigjon sjelljen e tij sipas distancës nga automjeti përpara.


In [ ]:

# Simulim i thjeshtë car-following për dy automjete:
# automjeti i parë lëviz me shpejtësi konstante,
# automjeti i dytë përshtatet sipas distancës.

dt = 0.05
T = 40
n_steps = int(T / dt)

t = np.linspace(0, T, n_steps)

x_lead = np.zeros(n_steps)
v_lead = np.ones(n_steps) * 18.0  # m/s

x_follow = np.zeros(n_steps)
v_follow = np.zeros(n_steps)

x_lead[0] = 20.0
x_follow[0] = 0.0
v_follow[0] = 10.0

a = 0.5
b = 0.08
v_des = 20.0
d_safe = 10.0

for n in range(n_steps - 1):
    x_lead[n + 1] = x_lead[n] + v_lead[n] * dt

    gap = x_lead[n] - x_follow[n]
    acc = a * (v_des - v_follow[n]) + b * (gap - d_safe)

    v_follow[n + 1] = max(0.0, v_follow[n] + acc * dt)
    x_follow[n + 1] = x_follow[n] + v_follow[n] * dt

plt.figure()
plt.plot(t, x_lead, label="Automjeti përpara", linewidth=2)
plt.plot(t, x_follow, label="Automjeti ndjekës", linewidth=2)
plt.xlabel("Koha")
plt.ylabel("Pozicioni")
plt.title("Model i thjeshtë car-following")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure()
plt.plot(t, v_lead, label="v_lead", linewidth=2)
plt.plot(t, v_follow, label="v_follow", linewidth=2)
plt.xlabel("Koha")
plt.ylabel("Shpejtësia")
plt.title("Shpejtësitë në modelin car-following")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



#### Koment

Ky model ilustron një ide thelbësore të sistemeve shumëagjentëshe:  
**rregulla lokale të ndërveprimit mund të prodhojnë dinamika kolektive jo triviale**.

Edhe kur modeli është shumë i thjeshtë, ai mund të prodhojë:

- afrime drejt regjimit të qëndrueshëm,
- oshilacione në distancë dhe shpejtësi,
- valë ngadalësimi në varg automjetesh.



### 4.2 Modeli Nagel–Schreckenberg

Modeli Nagel–Schreckenberg është një model diskret i trafikut, i ndërtuar si **automaton qelizor**.  
Rruga ndahet në qeliza diskrete dhe çdo automjet ka një shpejtësi të plotë

\[
v \in \{0,1,2,\dots,v_{\max}\}.
\]

Në çdo hap kohe zbatohen katër rregulla:

1. **Përshpejtimi:** nëse është e mundur, rritet shpejtësia.
2. **Frenimi nga afërsia:** shpejtësia kufizohet nga boshllëku përpara.
3. **Frenimi rastësor:** me probabilitet \(p\), shpejtësia zvogëlohet me 1.
4. **Lëvizja:** automjeti zhvendoset sipas shpejtësisë së re.

Modeli është shumë i njohur sepse tregon se si bllokimet mund të lindin spontanisht nga luhatje të vogla lokale.


In [ ]:

# Implementim i thjeshtë i modelit Nagel–Schreckenberg
L = 100              # gjatësia e rrugës (numri i qelizave)
density = 0.25       # dendësia e automjeteve
n_cars = int(L * density)
vmax = 5
p = 0.25
steps = 80

rng = np.random.default_rng(42)

# pozicionet fillestare pa mbivendosje
positions = np.sort(rng.choice(np.arange(L), size=n_cars, replace=False))
velocities = rng.integers(0, vmax + 1, size=n_cars)

history = np.full((steps, L), np.nan)

for step in range(steps):
    road = np.full(L, np.nan)
    for pos, vel in zip(positions, velocities):
        road[pos] = vel
    history[step] = road

    # llogarit boshllëkun përpara për çdo automjet (rrugë rrethore)
    gaps = np.empty(n_cars, dtype=int)
    for i in range(n_cars):
        if i < n_cars - 1:
            gaps[i] = positions[i + 1] - positions[i] - 1
        else:
            gaps[i] = (positions[0] + L) - positions[i] - 1

    # 1. përshpejtim
    velocities = np.minimum(velocities + 1, vmax)

    # 2. frenim për shkak të afërsisë
    velocities = np.minimum(velocities, gaps)

    # 3. frenim rastësor
    random_slow = (rng.random(n_cars) < p) & (velocities > 0)
    velocities[random_slow] -= 1

    # 4. lëvizja
    positions = (positions + velocities) % L

    # ruaj renditjen sipas pozicionit
    order = np.argsort(positions)
    positions = positions[order]
    velocities = velocities[order]

plt.figure(figsize=(10, 6))
plt.imshow(~np.isnan(history), aspect="auto", interpolation="nearest")
plt.xlabel("Pozicioni në rrugë")
plt.ylabel("Hapi kohor")
plt.title("Diagramë hapësirë-kohë për modelin Nagel–Schreckenberg")
plt.show()



#### Interpretim fizik

Në diagramën hapësirë-kohë, strukturat diagonale përfaqësojnë lëvizjen e automjeteve.  
Rajonet ku trajektoret grumbullohen tregojnë formimin e bllokimeve ose të rrjedhjes së ngadaltë.

Ky model është shembull i shkëlqyer i mënyrës se si:

- rregullat mikroskopike lokale,
- stokasticiteti i dobët,
- kufizimet hapësinore,

mund të prodhojnë **fenomene makroskopike kolektive**.



## 5. Simulime numerike dhe interpretim fizik

Të tre familjet e modeleve të trajtuara këtu, ndonëse i përkasin konteksteve të ndryshme, ndajnë disa ide qendrore.

### 5.1 Jolineariteti

- Në modelin logjistik, jolineariteti shfaqet përmes termit \(N(1-N/K)\).
- Në modelet epidemike, termi \(SI\) përfaqëson ndërveprimin jolinear midis klasave.
- Në trafik, kufizimet nga automjeti përpara krijojnë dinamika jo triviale.

### 5.2 Ekuilibri dhe stabiliteti

- Popullata logjistike tenton drejt kapacitetit mbajtës.
- Epidemia mund të shuhet ose të kalojë përmes një kulmi, në varësi të parametrave.
- Trafiku mund të kalojë nga rrjedhja e lirë në bllokim.

### 5.3 Sjellja kolektive emergjente

Ky është ndoshta mesazhi më i rëndësishëm i kësaj pjese:  
**sjellja makroskopike shpesh nuk është e dukshme drejtpërdrejt nga rregullat lokale**.

Kjo lidh drejtpërdrejt modelet biologjike dhe sociale me:

- fizikën statistikore,
- teorinë e sistemeve dinamike,
- shkencën e kompleksitetit.



## 6. Ushtrime për studentët

### Ushtrimi 1
Zgjidhni numerikisht modelin logjistik për disa vlera të ndryshme të \(r\) dhe \(K\).  
Analizoni si ndryshojnë:

- ritmi i rritjes fillestare,
- koha e afrimit drejt ekuilibrit,
- niveli i popullatës në regjim stacionar.

### Ushtrimi 2
Modifikoni modelin logjistik duke e bërë normën e rritjes sezonale:

\[
r(t) = r_0\left[1 + A\sin(\omega t)\right].
\]

Diskutoni si ndikon sezonaliteti në evolucionin e popullatës.

### Ushtrimi 3
Për modelin SIR, mbani \(\gamma\) konstant dhe ndryshoni \(\beta\).  
Gjeni për secilin rast:

- kulmin e \(I(t)\),
- kohën kur ndodh ky kulm,
- vlerën e \(R_0\).

### Ushtrimi 4
Krahasoni numerikisht modelet SIR dhe SEIR për të njëjtët parametra bazë.  
Shpjegoni pse klasa \(E\) ndryshon formën kohore të epidemisë.

### Ushtrimi 5
Në modelin car-following, ndryshoni parametrat \(a\), \(b\), \(v_{\mathrm{des}}\) dhe \(d\).  
Analizoni kur sistemi bëhet më i butë dhe kur shfaq luhatje më të forta.

### Ushtrimi 6
Zgjeroni modelin Nagel–Schreckenberg duke studiuar disa dendësi të ndryshme automjetesh.  
Për secilin rast, llogaritni shpejtësinë mesatare dhe diskutoni tranzicionin nga rrjedhja e lirë në bllokim.

### Ushtrimi 7
Shkruani një paragraf të shkurtër ku të shpjegoni pse këto modele, edhe pse jo mekanike në kuptimin klasik, mund të konsiderohen pjesë e natyrshme e kursit **Modelim në Fizikë**.



## 7. Përfundime

Në këtë notebook pamë se konceptet e modelimit fizik mund të transferohen me sukses në kontekste të tjera shkencore.  
Modelet demografike, epidemike dhe të trafikut ofrojnë shembuj shumë të qartë se si:

- zgjedhja e variablave të gjendjes,
- formulimi i rregullave të evolucionit,
- simulimi numerik,
- interpretimi i rezultateve,

bashkohen në një metodologji të vetme.

Në vijim, këto modele mund të zgjerohen në shumë drejtime:

- shtim i stokasticitetit,
- heterogjenitet i popullatës,
- rrjete kontakti jo uniforme,
- trafik me shumë korsi,
- kalim nga ODE në PDE ose modele agjent-bazuar.

Kjo e bën këtë temë një urë të fuqishme midis fizikës, biologjisë, shkencave sociale dhe shkencës së të dhënave.
